In [2]:
import langchain
from dotenv import load_dotenv
# import langchain_community
# import langchain_nvidia_ai_endpoints
# import langchain_pinecone

In [3]:
load_dotenv()

True

In [4]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
print(f"✅ LangChain version: {langchain.__version__}")

✅ LangChain version: 1.2.17


- Installation
- LLms
- prompt templates
- Chains
- Agent and Tools
- Memory
- Document Loader
- Indexes

In [6]:
# The correct, modern way to import
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os

load_dotenv()

nvidia_key = os.getenv("NVIDIA_API_KEY")


# 1. Initialize the LLM wrapper
# You can choose models like Llama 3.1, Mixtral, or Nemotron
llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct", 
    nvidia_api_key=nvidia_key,
    temperature=0.1,
)

print("✅ NVIDIA LLM Wrapper is ready to talk!")

✅ NVIDIA LLM Wrapper is ready to talk!


In [49]:
response = llm.invoke("What is a foreign key?")

In [50]:
# print(response.content)
print(response.response_metadata)

{'role': 'assistant', 'content': "**Foreign Key**\n================\n\nA foreign key is a field or column in a relational database table that refers to the primary key of another table. It is used to establish a relationship between two tables, allowing data to be linked and accessed across multiple tables.\n\n**Example**\n-----------\n\nSuppose we have two tables: `orders` and `customers`.\n\n`orders` table:\n\n| Order ID (Primary Key) | Customer ID (Foreign Key) | Order Date |\n| --- | --- | --- |\n| 1 | 1 | 2022-01-01 |\n| 2 | 1 | 2022-01-15 |\n| 3 | 2 | 2022-02-01 |\n\n`customers` table:\n\n| Customer ID (Primary Key) | Name | Address |\n| --- | --- | --- |\n| 1 | John Doe | 123 Main St |\n| 2 | Jane Smith | 456 Elm St |\n\nIn this example, the `Customer ID` column in the `orders` table is a foreign key that references the `Customer ID` primary key in the `customers` table. This establishes a relationship between the two tables, allowing us to link an order to the customer who plac

In [ ]:
%pip install langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

# llm_endpoint = HuggingFaceEndpoint(
#     repo_id="meta-llama/Llama-3.1-8B-Instruct",
#     task="text-generation",
#     huggingfacehub_api_token="hf_your_token_here"
# )
# chat_llm = ChatHuggingFace(llm=llm_endpoint)

##### Prompt Template

Think of a Prompt Template as a reusable form. Instead of typing the same instructions over and over, you create a structure with "blank spaces" (variables) that LangChain fills in automatically.

Once you have defined a Prompt Template, the real power comes from making it the first link in a Chain. In LangChain, a template isn't just a string; it’s a processing unit that prepares data for the AI.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Define the variable 'prompt_template'
# prompt_template = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful AI assistant."),
#     ("human", "{input}")
# ])

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional restaurant consultant specializing in {cuisine} culture."),
    ("human", "Give me 3 creative names for a new {cuisine} restaurant and a signature dish for each.")
])
prompt_template

ChatPromptTemplate(input_variables=['cuisine'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['cuisine'], input_types={}, partial_variables={}, template='You are a professional restaurant consultant specializing in {cuisine} culture.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['cuisine'], input_types={}, partial_variables={}, template='Give me 3 creative names for a new {cuisine} restaurant and a signature dish for each.'), additional_kwargs={})])

In [9]:


# The standard "Chain" formula: Prompt -> LLM -> Output
chain = prompt_template | llm | StrOutputParser()
# chain

# You only need to pass the variables; LangChain handles the rest
# response = chain.invoke({"input": "The Night's Watch"})
result = chain.invoke({"cuisine": "Indian"})
result

'Here are three creative name options for an Indian restaurant, along with a signature dish for each:\n\n1. **Restaurant Name: Tandoor Nights**\nSignature Dish: "Saffron Sunset" - A unique twist on the classic Chicken Tikka Masala, this dish features tender chicken marinated in a blend of saffron-infused yogurt, cumin, coriander, and cayenne pepper, grilled to perfection in a tandoor oven and served in a rich, creamy tomato sauce.\n\n2. **Restaurant Name: Dhaba Diaries**\nSignature Dish: "Punjabi Highway" - Inspired by the flavors of India\'s famous highway dhabas, this dish features tender lamb ribs slow-cooked in a rich, spicy curry made with a blend of Punjabi spices, including cumin, coriander, garam masala, and amchur powder, served with a side of fluffy basmati rice and crispy naan bread.\n\n3. **Restaurant Name: Spice Route Bazaar**\nSignature Dish: "Kerala Coconut Curry" - This dish takes inspiration from the flavors of Kerala, a state in southern India known for its rich cocon

In [11]:
from langchain_core.prompts import PromptTemplate

# 1. Define the template
# Note: Ensure "cuisine" is spelled the same in both places
prompt_template_name = PromptTemplate(
    input_variables=["cuisine"], 
    template="I want to open a restaurant for {cuisine} food."
)

# 2. Format the prompt (This creates the final string)
p = prompt_template_name.format(cuisine="Indian")

# 3. Check the prompt
print(p)

I want to open a restaurant for Indian food.


#### Chain

In LangChain, a Chain is essentially a "pipeline" that connects different components. Instead of running your prompt, then manually passing that to the LLM, and then manually cleaning the output, a Chain automates the entire flow.

Advanced Concept: The "RunnableParallel"
Sometimes you want to do two things at once. For example, you might want to ask the AI to "Summarize a text" AND "Translate it to Hindi" at the exact same time.

In [12]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate


# Creating two chains
summary_chain = ChatPromptTemplate.from_template("Summarize: {text} in 5 words") | llm
translate_chain = ChatPromptTemplate.from_template("Translate to Hindi: {text}") | llm

# Running them in parallel
map_chain = RunnableParallel(summary=summary_chain, translation=translate_chain)

# This returns a dictionary with both results!
results = map_chain.invoke({"text": "Quantum Computing is changing the world."})

In [13]:
results

{'summary': AIMessage(content='Revolutionizing technology with quantum power.', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'Revolutionizing technology with quantum power.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None, 'reasoning_content': None, 'token_usage': {'prompt_tokens': 50, 'total_tokens': 59, 'completion_tokens': 9, 'prompt_tokens_details': None}, 'finish_reason': 'stop', 'model_name': 'meta/llama-3.1-70b-instruct'}, id='lc_run--019e0b92-dff2-7b02-a83e-86b4b3f669eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 9, 'total_tokens': 59}, role='assistant'),
 'translation': AIMessage(content='क्वांटम कंप्यूटिंग दुनिया को बदल रही है।', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'क्वांटम कंप्यूटिंग दुनिया को बदल रही है।', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [

In [14]:
from langchain_core.prompts import PromptTemplate

# 1. Create the template
my_template = PromptTemplate.from_template("I am building a {project_name} using {tech_stack}.")

# 2. Format the prompt (This turns it into a simple string)
final_string = my_template.format(
    project_name="save eneregy", 
    tech_stack="NVIDIA and LangChain"
)

print(final_string)

I am building a save eneregy using NVIDIA and LangChain.


In [15]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. INITIALIZE THE BRAIN (LLM)
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0.1)


In [16]:
# 2. DEFINE PROMPT TEMPLATE #1 (Analysis)
analysis_prompt = ChatPromptTemplate.from_template(
    "Analyze these ingredients: {ingredients}. What is the single most concerning chemical?"
)

# 3. DEFINE PROMPT TEMPLATE #2 (Warning)
warning_prompt = ChatPromptTemplate.from_template(
    "You found {concern}. Explain why this is bad for a person's long-term health in 2 sentences."
)

In [17]:

# 4. COMBINE THEM INTO CHAINS
# Chain A: Ingredients -> Concern
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Chain B: The Full Sequence (Chain A's output becomes 'concern')
full_health_chain = (
    {"concern": analysis_chain} 
    | warning_prompt 
    | llm 
    | StrOutputParser()
)

# 5. RUN THE COMBINED ENGINE
final_report = full_health_chain.invoke({
    "ingredients": "Aspartame, Acesulfame Potassium, Phosphoric Acid"
})

print("--- PURE SCAN REPORT ---")
print(final_report)

--- PURE SCAN REPORT ---
Consuming aspartame regularly may lead to potential long-term health risks, including an increased risk of cancer, neurological disorders, and metabolic issues, such as insulin resistance and weight gain. Prolonged exposure to aspartame may also disrupt the balance of gut bacteria and affect glucose metabolism, potentially contributing to chronic diseases like diabetes and obesity.


In [18]:
final_report

'Consuming aspartame regularly may lead to potential long-term health risks, including an increased risk of cancer, neurological disorders, and metabolic issues, such as insulin resistance and weight gain. Prolonged exposure to aspartame may also disrupt the balance of gut bacteria and affect glucose metabolism, potentially contributing to chronic diseases like diabetes and obesity.'